In [ ]:
!pip install torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pytorch-lightning torchmetrics scikit-learn tqdm -q
!pip install aif360 -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision import transforms
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from tqdm.notebook import tqdm

# AIF360
from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE      = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE  = 64
NUM_WORKERS = 4
MAX_EPOCHS  = 3
THRESHOLD   = 0.57   # decision threshold — mirrors notebook 04

PROTECTED_ATTR = 'race'         # race column
GENDER_COL     = 'gender'       # gender column
LABEL_COL      = 'Cardiomegaly' # binary label  (1 = positive)
DICOM_ID_COL   = 'dicom_id'     # filename stem used to locate .npy files

RACE_DECODING   = {}
GENDER_DECODING = {0: 'M', 1: 'F', -1: 'Unknown'}

DATA_ROOT     = '/kaggle/input/notebooks/YOUR_USERNAME/01_data_preprocessing'
TRAIN_CSV     = os.path.join(DATA_ROOT, 'train.csv')
TEST_CSV      = os.path.join(DATA_ROOT, 'test.csv')
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train')   # {dir}/{label}/{dicom_id}.npy
VAL_IMG_DIR   = os.path.join(DATA_ROOT, 'test')    # {dir}/{label}/{dicom_id}.npy
BASELINE_CKPT = '/kaggle/input/datasets/YOUR_USERNAME/cardiomegaly-cnn-models/densenet121.ckpt'
OUTPUT_DIR    = '/kaggle/working/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Device: {DEVICE}')

In [ ]:
def load_file(path):
    return np.load(path).astype(np.float32)

train_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.472], [0.301]),
    transforms.RandomAffine(degrees=(-5, 5), translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomResizedCrop((224, 224), scale=(0.8, 1.0)),
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.472], [0.301]),
])

In [ ]:
# Dataset
class ChestXrayDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, meta_df, transform=None, sample_weights=None):
        self.img_dir        = img_dir
        self.records        = meta_df[[LABEL_COL, PROTECTED_ATTR, GENDER_COL]].copy()
        self.dicom_ids      = meta_df.index.tolist()
        self.transform      = transform
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.dicom_ids)

    def __getitem__(self, idx):
        dicom_id  = self.dicom_ids[idx]
        row       = self.records.iloc[idx]
        label     = int(row[LABEL_COL])
        protected = int(row[PROTECTED_ATTR])   # race int
        weight    = float(self.sample_weights[idx]) if self.sample_weights is not None else 1.0

        img_path = os.path.join(self.img_dir, str(label), f'{dicom_id}.npy')
        img = load_file(img_path)
        if self.transform:
            img = self.transform(img)
        return img, label, protected, weight

In [ ]:
# Model
class CardiomegalyModel(pl.LightningModule):
    """
    DenseNet-121 with a single binary output.
    When sample_weighted=True the per-sample loss is multiplied by the
    `weight` field returned by ChestXrayDataset (the Reweighing weights).
    """
    def __init__(self, pos_weight=4.6, sample_weighted=False):
        super().__init__()
        self.sample_weighted = sample_weighted
        self.model = torchvision.models.densenet121(weights='DEFAULT')
        self.model.features.conv0 = torch.nn.Conv2d(
            1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
        )
        self.model.classifier = torch.nn.Linear(1024, 1)
        self.register_buffer('pos_weight', torch.tensor([pos_weight]))
        self.loss_fn     = torch.nn.BCEWithLogitsLoss(pos_weight=self.pos_weight, reduction='none')
        self.train_auroc = torchmetrics.AUROC(task='binary')
        self.val_auroc   = torchmetrics.AUROC(task='binary')

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        x, y, _, w = batch
        y = y.float()
        logits = self(x).squeeze(-1)
        loss_per_sample = self.loss_fn(logits, y)
        loss = (loss_per_sample * w.to(self.device)).mean() if self.sample_weighted else loss_per_sample.mean()
        preds = torch.sigmoid(logits)
        auroc_metric = self.train_auroc if stage == 'train' else self.val_auroc
        auroc_metric(preds, y.int())
        self.log(f'{stage}_loss',  loss,         prog_bar=True)
        self.log(f'{stage}_auroc', auroc_metric, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):   return self._step(batch, 'train')
    def validation_step(self, batch, batch_idx): return self._step(batch, 'val')
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

In [ ]:
# Inference helper
def get_predictions(model, loader, device=DEVICE):
    """Returns (probs, labels, race_ints) numpy arrays."""
    model.eval()
    all_probs, all_labels, all_protected = [], [], []
    with torch.no_grad():
        for x, y, p, _ in tqdm(loader, desc='Inference'):
            logits = model(x.to(device)).squeeze(-1)
            all_probs.extend(torch.sigmoid(logits).cpu().numpy())
            all_labels.extend(y.numpy())
            all_protected.extend(p.numpy())
    return np.array(all_probs), np.array(all_labels), np.array(all_protected)

In [ ]:
# Metrics
DISPLAY_COLS = [
    'Overall', 'Sample (n)', 'Positive sample', 'Prevalence',
    'AUROC', 'AUPRC', 'Accuracy', 'Precision', 'Recall',
    'Specificity', 'F1', 'False positive rate', 'False negative rate',
]

def compute_group_metrics(probs, labels, protected, threshold=THRESHOLD, decoding=None):
    """
    Per-group classification metrics — formulas mirror notebook 04.

    Parameters
    ----------
    probs     : 1-D float array of predicted probabilities
    labels    : 1-D int array of ground-truth binary labels
    protected : 1-D int array of group codes
    threshold : decision threshold (default = THRESHOLD = 0.57)
    decoding  : dict mapping int codes → display strings
    """
    rows = []
    y_pred_all = (probs >= threshold).astype(int)

    for g in np.unique(protected):
        mask = protected == g
        yt, ypr, yp = labels[mask], probs[mask], y_pred_all[mask]
        n     = len(yt)
        n_pos = int(yt.sum())
        if n == 0:
            continue

        cm = confusion_matrix(yt, yp, labels=[0, 1])
        TN, FP, FN, TP = cm.ravel() if cm.shape == (2, 2) else (cm[0, 0], 0, 0, 0)

        precision   = TP / (TP + FP + 1e-9)
        recall      = TP / (TP + FN + 1e-9)
        specificity = TN / (TN + FP + 1e-9)
        fpr         = FP / (FP + TN + 1e-9)
        fnr         = FN / (FN + TP + 1e-9)
        f1          = 2 * precision * recall / (precision + recall + 1e-9)
        accuracy    = (TP + TN) / n

        if n_pos > 0 and n_pos < n:
            auroc = roc_auc_score(yt, ypr)
            auprc = average_precision_score(yt, ypr)
        else:
            auroc = auprc = float('nan')

        group_name = decoding.get(int(g), str(int(g))) if decoding else str(int(g))
        rows.append({
            'Overall'             : group_name,
            'Sample (n)'          : n,
            'Positive sample'     : n_pos,
            'Prevalence'          : round(n_pos / n,   4),
            'AUROC'               : round(auroc,       4),
            'AUPRC'               : round(auprc,       4),
            'Accuracy'            : round(accuracy,    4),
            'Precision'           : round(precision,   4),
            'Recall'              : round(recall,      4),
            'Specificity'         : round(specificity, 4),
            'F1'                  : round(f1,          4),
            'False positive rate' : round(fpr,         4),
            'False negative rate' : round(fnr,         4),
            'FPR': round(fpr, 4), 'FNR': round(fnr, 4),
            'N': n, 'Pos': n_pos, 'AUROC_raw': auroc,
        })

    df = pd.DataFrame(rows)
    df['EO_Gap (FNR)'] = df['FNR'].max() - df['FNR'].min()
    return df

In [ ]:
# Training helper
def train_model(model, train_loader, val_loader, save_dir, max_epochs=MAX_EPOCHS):
    os.makedirs(save_dir, exist_ok=True)
    ckpt_cb = ModelCheckpoint(
        monitor='val_auroc', dirpath=save_dir,
        filename='{epoch}-{val_auroc:.3f}',
        save_top_k=1, mode='max'
    )
    trainer = pl.Trainer(
        accelerator='gpu', devices=1,
        logger=TensorBoardLogger(save_dir='./logs'),
        callbacks=[ckpt_cb], max_epochs=max_epochs, log_every_n_steps=1
    )
    trainer.fit(model, train_loader, val_loader)
    return ckpt_cb.best_model_path

In [ ]:
# AIF360 helper
def build_aif360_dataset(meta_df, protected_col):
    aif_df = (
        meta_df[[protected_col, LABEL_COL]]
        .astype(float)
        .reset_index(drop=True)   # drop dicom_id from the DataFrame
    )
    return BinaryLabelDataset(
        df=aif_df,
        label_names=[LABEL_COL],
        protected_attribute_names=[protected_col],
        favorable_label=1.0,
        unfavorable_label=0.0,
    )


def apply_reweighing(meta_df, protected_col, privileged_val, unprivileged_val):
    privileged_groups   = [{protected_col: float(privileged_val)}]
    unprivileged_groups = [{protected_col: float(unprivileged_val)}]

    aif_ds = build_aif360_dataset(meta_df, protected_col)
    RW = Reweighing(
        privileged_groups=privileged_groups,
        unprivileged_groups=unprivileged_groups
    )
    aif_ds_rw = RW.fit_transform(aif_ds)

    # instance_weights is aligned to the input row order (reset_index(drop=True))
    sample_weights = aif_ds_rw.instance_weights.flatten()
    print(f'Weight stats for {protected_col}:')
    print(pd.Series(sample_weights).describe().to_string())
    return sample_weights

In [ ]:
# Load train / test CSVs
train_meta = pd.read_csv(TRAIN_CSV)
val_meta   = pd.read_csv(TEST_CSV)

train_meta.columns = train_meta.columns.str.strip()
val_meta.columns   = val_meta.columns.str.strip()

train_meta = train_meta.set_index(DICOM_ID_COL)
val_meta   = val_meta.set_index(DICOM_ID_COL)

# Race encoding
# White → 0 (privileged), all other races sorted alphabetically → 1, 2, …
# Derived from combined train + val to ensure consistency across splits.
_all_races   = sorted(pd.concat([train_meta, val_meta])[PROTECTED_ATTR].dropna().unique().tolist())
_other_races = [r for r in _all_races if r != 'White']
_ordered     = (['White'] if 'White' in _all_races else []) + _other_races
RACE_ENCODING = {name: i for i, name in enumerate(_ordered)}
RACE_DECODING.update({i: name for name, i in RACE_ENCODING.items()})
print('Race encoding:', RACE_ENCODING)

for _df in [train_meta, val_meta]:
    _df[PROTECTED_ATTR] = _df[PROTECTED_ATTR].map(RACE_ENCODING).fillna(-1).astype(int)

# Gender encoding (binary)
GENDER_ENCODING = {'M': 0, 'Male': 0, 'male': 0, 'F': 1, 'Female': 1, 'female': 1}
GENDER_DECODING.update({0: 'M', 1: 'F', -1: 'Unknown'})

for _df in [train_meta, val_meta]:
    _df[GENDER_COL] = _df[GENDER_COL].map(GENDER_ENCODING).fillna(-1).astype(int)

# Binarised race column for AIF360
# AIF360 Reweighing requires a BINARY protected attribute
# (one privileged group vs one unprivileged group).
# race_bin: White=0 (privileged), non-White=1 (unprivileged).
# Unknown race (-1) is excluded from reweighing weight computation.
RACE_BIN_COL = 'race_bin'
for _df in [train_meta, val_meta]:
    _df[RACE_BIN_COL] = _df[PROTECTED_ATTR].apply(
        lambda v: 0 if v == 0 else (1 if v > 0 else -1)
    )

print(f'\nTrain: {len(train_meta):,} samples  |  Test: {len(val_meta):,} samples')
print('\nRace distribution (train):')
print(train_meta[PROTECTED_ATTR].map(RACE_DECODING).value_counts().to_string())
print('\nGender distribution (train):')
print(train_meta[GENDER_COL].map(GENDER_DECODING).value_counts().to_string())
print('\nLabel distribution (train):')
print(train_meta[LABEL_COL].value_counts().to_string())


In [ ]:
baseline_model = CardiomegalyModel.load_from_checkpoint(BASELINE_CKPT)
baseline_model.to(DEVICE)
baseline_model.freeze()

val_ds_base = ChestXrayDataset(VAL_IMG_DIR, val_meta, transform=val_transforms)
val_loader_base = torch.utils.data.DataLoader(
    val_ds_base, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False
)

probs_base, labels_base, protected_base = get_predictions(baseline_model, val_loader_base)
gender_base = val_meta[GENDER_COL].values

# Race metrics
race_metrics_base = compute_group_metrics(
    probs_base, labels_base, protected_base, decoding=RACE_DECODING
)
print('=== Baseline — Race Subgroup Metrics ===')
print(race_metrics_base[DISPLAY_COLS].to_string(index=False))

# Gender metrics
gender_metrics_base = compute_group_metrics(
    probs_base, labels_base, gender_base, decoding=GENDER_DECODING
)
print('\n=== Baseline — Gender Subgroup Metrics ===')
print(gender_metrics_base[DISPLAY_COLS].to_string(index=False))

race_metrics_base.insert(0,   'Method', 'Baseline')
gender_metrics_base.insert(0, 'Method', 'Baseline')

In [ ]:
# Compute Reweighing weights for race
# Use only samples with known race for the AIF360 fit.
train_meta_known_race = train_meta[train_meta[RACE_BIN_COL] != -1].copy()

weights_race_known = apply_reweighing(
    train_meta_known_race,
    protected_col=RACE_BIN_COL,
    privileged_val=0,       # White
    unprivileged_val=1,     # non-White
)

# Align weights back to the full train_meta row order.
# Rows absent from train_meta_known_race (unknown race) get weight 1.0.
weight_series_race = pd.Series(
    weights_race_known, index=train_meta_known_race.index
)
sample_weights_race = (
    train_meta.index
    .map(weight_series_race)
    .fillna(1.0)
    .to_numpy(dtype=float)
)

print(f'\nWeights aligned to full train_meta ({len(sample_weights_race):,} samples).')
print('Weight distribution (all rows):')
print(pd.Series(sample_weights_race).describe().to_string())

In [ ]:
# Train with race-reweighing
train_ds_race_rw = ChestXrayDataset(
    TRAIN_IMG_DIR, train_meta,
    transform=train_transforms,
    sample_weights=sample_weights_race,
)
train_loader_race_rw = torch.utils.data.DataLoader(
    train_ds_race_rw, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True
)

model_race_rw = CardiomegalyModel(sample_weighted=True)
ckpt_race_rw  = train_model(
    model_race_rw, train_loader_race_rw, val_loader_base,
    save_dir=os.path.join(OUTPUT_DIR, 'reweighing_race')
)
print(f'Best checkpoint: {ckpt_race_rw}')

In [ ]:
# Evaluate race-reweighed model
model_race_rw = CardiomegalyModel.load_from_checkpoint(ckpt_race_rw).to(DEVICE)
model_race_rw.freeze()

probs_race_rw, labels_race_rw, protected_race_rw = get_predictions(
    model_race_rw, val_loader_base
)
gender_race_rw = val_meta[GENDER_COL].values

race_metrics_rw = compute_group_metrics(
    probs_race_rw, labels_race_rw, protected_race_rw, decoding=RACE_DECODING
)
print('=== Reweighing (Race) — Race Subgroup Metrics ===')
print(race_metrics_rw[DISPLAY_COLS].to_string(index=False))

gender_metrics_rw_race = compute_group_metrics(
    probs_race_rw, labels_race_rw, gender_race_rw, decoding=GENDER_DECODING
)
print('\n=== Reweighing (Race) — Gender Subgroup Metrics ===')
print(gender_metrics_rw_race[DISPLAY_COLS].to_string(index=False))

race_metrics_rw.insert(0,        'Method', 'Reweighing (Race)')
gender_metrics_rw_race.insert(0, 'Method', 'Reweighing (Race)')